# 项目五：半监督学习（介于监督和无监督之间）

**目标**：用「少量有标签 + 大量无标签」的数据，做出接近「全部有标签」的效果。

## 为什么需要半监督？—— 因为「标注」很贵

回想前面的项目：

- **监督学习**：每个数据都要有标签。但标签是**人工一个个标出来的**——比如手写数字，要有人一张张看「这是 3、这是 7」。1000 张图要标 1000 次，贵。
- **无监督学习**：不要标签，但它只能「分堆」，没法直接告诉你「这是数字几」。

**半监督学习的思路**：现实中「未标注数据」要多少有多少（网上随便抓），但「标注」很贵。那能不能：**只标注一小部分，剩下的大部分不标，两个一起用？**

答案是可以，而且效果惊人——这就是半监督学习。

## 核心思想：标签会「传播」

半监督学习基于一个朴素假设：**「长得像」的数据，标签应该也一样**。

算法叫 **LabelPropagation（标签传播）**，想象：

- 每个数据点是一个节点，相近的点之间连一条边（形成一张图）
- 有标签的点是「墨水滴」，墨水顺着边**扩散**到周围的无标签点
- 越近的邻居，被「染」得越深

于是，那些没标签的点，通过「和谁长得像」，被自动贴上了标签。

> 下面用你熟悉的手写数字来验证：只标 10%，剩下 90% 让算法自己「传播」出标签。

In [ ]:
# 第 1 步：加载数据 + 构造「半监督」数据集（抹掉 90% 的标签）
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

digits = load_digits()
X = digits.data
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 关键：模拟「标注很贵」的现实 —— 只保留 10% 的标签，其余 90% 抹掉
rng = np.random.RandomState(42)
n_labeled = int(len(y_train) * 0.10)   # 只标 10%
labeled_idx = rng.choice(len(y_train), n_labeled, replace=False)

y_train_masked = np.full(len(y_train), -1)     # 先全部设成 -1（-1 = 未标注）
y_train_masked[labeled_idx] = y_train[labeled_idx]   # 只恢复 10% 的真实标签

print(f"训练集共 {len(y_train)} 张图")
print(f"有标签的: {(y_train_masked != -1).sum()} 张（10%）")
print(f"没标签的: {(y_train_masked == -1).sum()} 张（90%）")

In [ ]:
# 第 2 步：Baseline —— 只用那 10% 有标签的数据做普通监督学习
# 这是「穷人家的做法」：没钱标注，只能拿手上这 10% 硬学
X_labeled = X_train[labeled_idx]
y_labeled = y_train[labeled_idx]

svm = SVC()
svm.fit(X_labeled, y_labeled)
baseline_acc = accuracy_score(y_test, svm.predict(X_test))

print(f"只用 10% 标签（监督学习）: {baseline_acc:.2%}")

## 半监督的做法：10% 有标签 + 90% 无标签 一起喂进去

注意下面 `fit` 传进去的 `y_train_masked`：里面大部分是 **-1（未标注）**。

`LabelPropagation` 会自己处理这些 -1：用「标签传播」给它们贴标签。

> 我们用 `kernel='knn'`（K 近邻核），因为默认的 RBF 核在高维数据上会失效（你以后会学到为什么）。

In [ ]:
# 第 3 步：半监督学习 —— 两种算法都试
lp = LabelPropagation(kernel="knn", n_neighbors=7, max_iter=2000)
lp.fit(X_train, y_train_masked)   # 注意：喂的是「大部分是 -1」的标签
lp_acc = accuracy_score(y_test, lp.predict(X_test))

ls = LabelSpreading(kernel="knn", n_neighbors=7, max_iter=2000)
ls.fit(X_train, y_train_masked)
ls_acc = accuracy_score(y_test, ls.predict(X_test))

print(f"LabelPropagation（半监督）: {lp_acc:.2%}")
print(f"LabelSpreading  （半监督）: {ls_acc:.2%}")

In [ ]:
# 第 4 步：三组对比，看半监督的价值
full_svm = SVC().fit(X_train, y_train)   # 用全部 100% 标签的「上限」
full_acc = accuracy_score(y_test, full_svm.predict(X_test))

print("=" * 46)
print(f"只用 10% 标签（监督）            : {baseline_acc:.2%}")
print(f"半监督（10% 标注 + 90% 无标注）   : {ls_acc:.2%}")
print(f"用全部 100% 标签（理想上限）      : {full_acc:.2%}")
print("=" * 46)

# 画个柱状图更直观
names = ["只用10%标签", "半监督", "全部100%标签"]
vals = [baseline_acc, ls_acc, full_acc]
plt.figure(figsize=(7, 5))
bars = plt.bar(names, vals, color=["#d9534f", "#5cb85c", "#5bc0de"])
plt.ylabel("准确率")
plt.title("半监督学习：用便宜的无标注数据逼近上限")
for bar, v in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.2%}",
             ha="center", fontsize=11)
plt.ylim(0.8, 1.02)
plt.show()

## 自己动手改（练习）

1. 把 `n_labeled` 的 `0.10` 改成 `0.05`（只标 5%）或 `0.20`（标 20%），重新跑，看「半监督相对 baseline 的提升」是变大还是变小？——想想：标注越少的时候，半监督是不是越有用？
2. **核心理解题**：试着用自己的话说说，为什么「没标签的数据」也能帮忙提升准确率？（提示：回想「标签传播」的比喻——没标签的点靠什么被贴上了标签？）
3. 观察 `LabelPropagation` 和 `LabelSpreading` 两个的结果，谁更好一点？

## 学到这里，你已经集齐了三种学习方式

| 方式 | 有没有答案 | 典型任务 | 你做的项目 |
|---|---|---|---|
| 监督 | 全都有 | 分类、回归 | 鸢尾花、手写数字、泰坦尼克 |
| 无监督 | 全都没有 | 聚类 | KMeans 分堆 |
| **半监督** | **一小部分有** | 两者之间 | 本项目的标签传播 |

**恭喜，机器学习最基础的三种范式，你都亲手跑通了。**